# PineBrook IGNITE Prize Wheel

Run the setup cell once, then run the wheel code cell and usage cell.

What this version includes:
- Class dropdown above the student name with classes K1-K5.
- Student name entry on the left of the wheel.
- Click the center of the wheel to spin.
- Prize tracker on the right with editable K dropdowns, student names, prizes, and overwrite controls.
- Local saving to `student_prize_tracker.json`.
- Automatic loading from the saved local file when the notebook starts.
- One saved prize per student per class. If the same student in the same class plays again, their prize is replaced.
- Two clear buttons:
  - **Clear Current List** clears only the current on-screen list.
  - **Clear Entire List** requires two clicks and clears the on-screen list plus the saved local JSON file.


In [ ]:
%pip install anywidget ipywidgets pandas


In [ ]:
from __future__ import annotations

import json
import random
import secrets
from datetime import datetime
from pathlib import Path
from typing import Iterable

import anywidget
import traitlets
from IPython.display import display


class StudentPrizeWheel(anywidget.AnyWidget):
    """A classroom prize wheel with class-based student prize tracking.

    What it does:
    - Lets you choose a class (K1-K5) above the student name.
    - Lets a student type their name on the left.
    - Lets them click SPIN in the middle.
    - Chooses a prize from the list you pass in.
    - Tracks each student's class, name, and prize on the right.
    - Saves the tracker locally as JSON and reloads it when the notebook runs again.
    - Keeps exactly one saved prize per student per class; repeat spins replace the previous prize.
    - Lets you clear only the current in-memory list, or clear the full saved list.
    - Lets you overwrite a student's prize from the tracker.
    """

    _css = r"""
    .tp-wheel-app {
      width: min(1220px, 100%);
      margin: 16px auto;
      display: grid;
      grid-template-columns: minmax(220px, 260px) minmax(360px, 1fr) minmax(330px, 430px);
      gap: 18px;
      align-items: start;
      font-family: system-ui, -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif;
      color: #111827;
    }

    .tp-panel {
      background: #ffffff;
      border: 1px solid #e5e7eb;
      border-radius: 18px;
      padding: 16px;
      box-shadow: 0 8px 22px rgba(17, 24, 39, 0.08);
    }

    .tp-panel-title {
      font-size: 1.05rem;
      font-weight: 850;
      margin-bottom: 10px;
    }

    .tp-label {
      display: block;
      font-size: 0.88rem;
      font-weight: 750;
      margin: 10px 0 6px;
    }

    .tp-input, .tp-select {
      width: 100%;
      box-sizing: border-box;
      border: 1px solid #cbd5e1;
      border-radius: 12px;
      padding: 10px 12px;
      font-size: 0.96rem;
      outline: none;
      background: #ffffff;
    }

    .tp-input:focus, .tp-select:focus {
      border-color: #2563eb;
      box-shadow: 0 0 0 3px rgba(37, 99, 235, 0.18);
    }

    .tp-help {
      margin-top: 10px;
      font-size: 0.86rem;
      line-height: 1.35;
      color: #4b5563;
    }

    .tp-status {
      margin-top: 12px;
      padding: 10px;
      min-height: 20px;
      border-radius: 12px;
      background: #f8fafc;
      font-size: 0.9rem;
      font-weight: 650;
      line-height: 1.35;
      white-space: pre-line;
      color: #1f2937;
    }

    .tp-wheel-shell {
      display: grid;
      place-items: center;
      gap: 12px;
      width: 100%;
    }

    .tp-wheel-title {
      font-size: 1.45rem;
      font-weight: 900;
      text-align: center;
    }

    .tp-wheel-stage {
      position: relative;
      width: min(520px, 92vw);
      aspect-ratio: 1;
      display: grid;
      place-items: center;
    }

    .tp-wheel-spinner {
      width: 100%;
      height: 100%;
      border-radius: 50%;
      overflow: hidden;
      background: #ffffff;
      box-shadow: 0 14px 35px rgba(17, 24, 39, 0.22);
      will-change: transform;
    }

    .tp-wheel-spinner canvas {
      display: block;
      width: 100%;
      height: 100%;
    }

    .tp-wheel-pointer {
      position: absolute;
      top: -3px;
      left: 50%;
      transform: translateX(-50%);
      width: 0;
      height: 0;
      border-left: 20px solid transparent;
      border-right: 20px solid transparent;
      border-top: 38px solid #dc2626;
      filter: drop-shadow(0 3px 2px rgba(0, 0, 0, 0.25));
      z-index: 4;
    }

    .tp-wheel-button {
      position: absolute;
      width: min(120px, 24vw);
      aspect-ratio: 1;
      border-radius: 999px;
      border: 4px solid #111827;
      background: #ffffff;
      color: #111827;
      font-weight: 900;
      letter-spacing: 0.04em;
      cursor: pointer;
      z-index: 5;
      box-shadow: 0 5px 18px rgba(0, 0, 0, 0.22);
    }

    .tp-wheel-button:hover:not(:disabled) {
      transform: scale(1.03);
    }

    .tp-wheel-button:disabled {
      cursor: wait;
      opacity: 0.72;
    }

    .tp-wheel-result {
      min-height: 2rem;
      font-size: 1.25rem;
      font-weight: 850;
      text-align: center;
    }

    .tp-wheel-hint {
      font-size: 0.9rem;
      color: #4b5563;
      text-align: center;
    }

    .tp-tracker-actions {
      display: grid;
      gap: 8px;
      margin-bottom: 12px;
    }

    .tp-button-row {
      display: flex;
      gap: 8px;
      flex-wrap: wrap;
      align-items: center;
    }

    .tp-action-button {
      border: none;
      border-radius: 12px;
      padding: 9px 11px;
      font-weight: 800;
      cursor: pointer;
      background: #111827;
      color: #ffffff;
    }

    .tp-action-button.secondary {
      background: #2563eb;
    }

    .tp-action-button.neutral {
      background: #4b5563;
    }

    .tp-action-button.danger {
      background: #dc2626;
    }

    .tp-action-button:disabled {
      opacity: 0.55;
      cursor: not-allowed;
    }

    .tp-table-wrap {
      max-height: 460px;
      overflow: auto;
      border: 1px solid #e5e7eb;
      border-radius: 14px;
    }

    .tp-table {
      width: 100%;
      border-collapse: collapse;
      font-size: 0.86rem;
    }

    .tp-table th,
    .tp-table td {
      padding: 9px;
      border-bottom: 1px solid #e5e7eb;
      vertical-align: middle;
      text-align: left;
    }

    .tp-table th {
      position: sticky;
      top: 0;
      background: #f8fafc;
      font-weight: 850;
      z-index: 1;
    }

    .tp-table tr:last-child td {
      border-bottom: none;
    }

    .tp-k-cell {
      font-weight: 900;
      text-align: center;
      width: 38px;
    }

    .tp-row-select {
      width: 100%;
      min-width: 130px;
      border: 1px solid #cbd5e1;
      border-radius: 10px;
      padding: 6px;
      background: #ffffff;
      margin-bottom: 6px;
    }

    .tp-small-button {
      border: none;
      border-radius: 10px;
      padding: 7px 9px;
      font-weight: 800;
      cursor: pointer;
      background: #2563eb;
      color: #ffffff;
    }

    .tp-empty {
      padding: 14px;
      color: #6b7280;
      font-style: italic;
    }

    @media (max-width: 980px) {
      .tp-wheel-app {
        grid-template-columns: 1fr;
      }
      .tp-wheel-stage {
        width: min(500px, 92vw);
      }
    }
    """

    _esm = r"""
    const COLORS = [
      "#fecaca", "#fed7aa", "#fde68a", "#bbf7d0",
      "#bfdbfe", "#ddd6fe", "#fbcfe8", "#ccfbf1",
      "#e9d5ff", "#d9f99d", "#fef3c7", "#bae6fd"
    ];

    function cleanText(value) {
      return String(value ?? "").trim().replace(/\s+/g, " ");
    }

    function cleanClassLabel(value, classOptions) {
      let label = cleanText(value).toUpperCase().replace(/\s+/g, "");
      if (/^\d+$/.test(label)) {
        label = `K${label}`;
      } else if (/^K\d+$/.test(label)) {
        label = `K${label.slice(1)}`;
      }
      return classOptions.includes(label) ? label : (classOptions[0] || "K1");
    }

    function classNumber(value) {
      return cleanText(value).replace(/^K/i, "");
    }

    function shortLabel(label, maxLength = 24) {
      label = cleanText(label);
      return label.length > maxLength ? label.slice(0, maxLength - 1) + "..." : label;
    }

    function drawWheel(canvas, prizes) {
      const dpr = window.devicePixelRatio || 1;
      const rect = canvas.getBoundingClientRect();
      const size = Math.max(320, Math.floor(rect.width || 520));

      canvas.width = Math.floor(size * dpr);
      canvas.height = Math.floor(size * dpr);

      const ctx = canvas.getContext("2d");
      ctx.setTransform(dpr, 0, 0, dpr, 0, 0);
      ctx.clearRect(0, 0, size, size);

      const cx = size / 2;
      const cy = size / 2;
      const radius = size * 0.48;
      const innerRadius = size * 0.13;
      const count = prizes.length;
      const slice = (Math.PI * 2) / count;

      prizes.forEach((prize, index) => {
        const start = -Math.PI / 2 + index * slice;
        const end = start + slice;

        ctx.beginPath();
        ctx.moveTo(cx, cy);
        ctx.arc(cx, cy, radius, start, end);
        ctx.closePath();
        ctx.fillStyle = COLORS[index % COLORS.length];
        ctx.fill();

        ctx.strokeStyle = "#111827";
        ctx.lineWidth = 2;
        ctx.stroke();

        ctx.save();
        ctx.translate(cx, cy);
        ctx.rotate(start + slice / 2);
        ctx.textAlign = "right";
        ctx.textBaseline = "middle";
        ctx.fillStyle = "#111827";
        ctx.font = `700 ${Math.max(11, Math.min(17, size / 34))}px system-ui, sans-serif`;
        ctx.fillText(shortLabel(prize), radius - 18, 0);
        ctx.restore();
      });

      ctx.beginPath();
      ctx.arc(cx, cy, innerRadius, 0, Math.PI * 2);
      ctx.fillStyle = "#ffffff";
      ctx.fill();
      ctx.lineWidth = 3;
      ctx.strokeStyle = "#111827";
      ctx.stroke();
    }

    function normalizeDegrees(value) {
      return ((value % 360) + 360) % 360;
    }

    function optionList(prizes, selected) {
      const cleanSelected = cleanText(selected);
      const options = [];

      if (cleanSelected && !prizes.includes(cleanSelected)) {
        const savedOption = document.createElement("option");
        savedOption.value = cleanSelected;
        savedOption.textContent = `${cleanSelected} (saved)`;
        savedOption.selected = true;
        options.push(savedOption);
      }

      prizes.forEach((prize) => {
        const option = document.createElement("option");
        option.value = prize;
        option.textContent = prize;
        option.selected = prize === cleanSelected;
        options.push(option);
      });

      return options;
    }

    function render({ model, el }) {
      let rotation = 0;
      let locked = false;
      let fallbackTimer = null;
      let clearAllArmed = false;
      let clearAllTimer = null;

      const app = document.createElement("div");
      app.className = "tp-wheel-app";

      // Left panel: class and student entry
      const left = document.createElement("section");
      left.className = "tp-panel";

      const leftTitle = document.createElement("div");
      leftTitle.className = "tp-panel-title";
      leftTitle.textContent = "Student";

      const classLabel = document.createElement("label");
      classLabel.className = "tp-label";
      classLabel.textContent = "Class";

      const classSelect = document.createElement("select");
      classSelect.className = "tp-select";

      const classOptions = model.get("class_options") || ["K1", "K2", "K3", "K4", "K5"];
      const selectedClass = cleanClassLabel(model.get("selected_class"), classOptions);
      classOptions.forEach((className) => {
        const option = document.createElement("option");
        option.value = className;
        option.textContent = className;
        option.selected = className === selectedClass;
        classSelect.appendChild(option);
      });

      const nameLabel = document.createElement("label");
      nameLabel.className = "tp-label";
      nameLabel.textContent = "Student name";

      const nameInput = document.createElement("input");
      nameInput.className = "tp-input";
      nameInput.type = "text";
      nameInput.placeholder = "Type student name";
      nameInput.value = model.get("current_student") || "";

      const leftHelp = document.createElement("div");
      leftHelp.className = "tp-help";
      leftHelp.textContent = "Choose the class first, then type the student's name. If a K is entered wrong, fix it from the K dropdown in the tracker.";

      const saveInfo = document.createElement("div");
      saveInfo.className = "tp-help";
      saveInfo.textContent = `Saved locally to: ${model.get("storage_path") || "student_prize_tracker.json"}`;

      const status = document.createElement("div");
      status.className = "tp-status";
      status.setAttribute("aria-live", "polite");
      status.textContent = model.get("status_message") || "Ready.";

      left.append(leftTitle, classLabel, classSelect, nameLabel, nameInput, leftHelp, saveInfo, status);

      // Middle panel: wheel
      const middle = document.createElement("main");
      middle.className = "tp-wheel-shell";

      const title = document.createElement("div");
      title.className = "tp-wheel-title";
      title.textContent = model.get("title") || "PineBrook IGNITE Prize Wheel";

      const stage = document.createElement("div");
      stage.className = "tp-wheel-stage";

      const pointer = document.createElement("div");
      pointer.className = "tp-wheel-pointer";
      pointer.setAttribute("aria-hidden", "true");

      const spinner = document.createElement("div");
      spinner.className = "tp-wheel-spinner";

      const canvas = document.createElement("canvas");
      spinner.appendChild(canvas);

      const button = document.createElement("button");
      button.className = "tp-wheel-button";
      button.type = "button";
      button.textContent = "SPIN";
      button.setAttribute("aria-label", "Spin the classroom prize wheel");

      const result = document.createElement("div");
      result.className = "tp-wheel-result";
      result.setAttribute("aria-live", "polite");

      const hint = document.createElement("div");
      hint.className = "tp-wheel-hint";
      hint.textContent = "Click SPIN in the middle of the wheel.";

      stage.append(pointer, spinner, button);
      middle.append(title, stage, result, hint);

      // Right panel: tracker
      const right = document.createElement("section");
      right.className = "tp-panel";

      const trackerTitle = document.createElement("div");
      trackerTitle.className = "tp-panel-title";
      trackerTitle.textContent = "Prize Tracker";

      const actions = document.createElement("div");
      actions.className = "tp-tracker-actions";

      const manualLabel = document.createElement("label");
      manualLabel.className = "tp-label";
      manualLabel.textContent = "Set or overwrite current student";

      const manualSelect = document.createElement("select");
      manualSelect.className = "tp-select";

      const actionRow = document.createElement("div");
      actionRow.className = "tp-button-row";

      const overwriteCurrentButton = document.createElement("button");
      overwriteCurrentButton.className = "tp-action-button secondary";
      overwriteCurrentButton.type = "button";
      overwriteCurrentButton.textContent = "Set / Overwrite";

      const clearCurrentButton = document.createElement("button");
      clearCurrentButton.className = "tp-action-button neutral";
      clearCurrentButton.type = "button";
      clearCurrentButton.textContent = "Clear Current List";

      const clearAllButton = document.createElement("button");
      clearAllButton.className = "tp-action-button danger";
      clearAllButton.type = "button";
      clearAllButton.textContent = "Clear Entire List";

      actionRow.append(overwriteCurrentButton, clearCurrentButton, clearAllButton);
      actions.append(manualLabel, manualSelect, actionRow);

      const clearHelp = document.createElement("div");
      clearHelp.className = "tp-help";
      clearHelp.textContent = "Clear Current List only clears what is shown right now. Clear Entire List needs two clicks and also clears the saved local file.";

      const tableWrap = document.createElement("div");
      tableWrap.className = "tp-table-wrap";

      right.append(trackerTitle, actions, clearHelp, tableWrap);
      app.append(left, middle, right);
      el.replaceChildren(app);

      function getPrizes() {
        return (model.get("prizes") || []).map(cleanText).filter(Boolean);
      }

      function getAssignments() {
        return model.get("assignments") || [];
      }

      function updateManualSelect() {
        const selected = manualSelect.value || getPrizes()[0] || "";
        manualSelect.replaceChildren(...optionList(getPrizes(), selected));
      }

      function updateButton() {
        const prizes = getPrizes();
        const spinning = Boolean(model.get("spinning"));
        const hasName = Boolean(cleanText(nameInput.value));
        button.disabled = locked || spinning || prizes.length < 2 || !hasName;
        button.textContent = locked || spinning ? "..." : "SPIN";
        overwriteCurrentButton.disabled = prizes.length === 0 || !hasName;
        clearCurrentButton.disabled = getAssignments().length === 0;
        // Clear Entire List stays enabled so it can clear the saved file even after the current list is cleared.
        clearAllButton.disabled = false;
      }

      function redraw() {
        const prizes = getPrizes();
        if (prizes.length >= 2) {
          drawWheel(canvas, prizes);
          hint.textContent = cleanText(nameInput.value)
            ? "Click SPIN in the middle of the wheel."
            : "Enter a student name before spinning.";
        } else {
          hint.textContent = "Add at least two prizes to spin the wheel.";
        }
        updateManualSelect();
        updateButton();
      }

      function renderTable() {
        const assignments = getAssignments();
        tableWrap.replaceChildren();

        if (!assignments.length) {
          const empty = document.createElement("div");
          empty.className = "tp-empty";
          empty.textContent = "No students tracked yet.";
          tableWrap.appendChild(empty);
          updateButton();
          return;
        }

        const table = document.createElement("table");
        table.className = "tp-table";

        const thead = document.createElement("thead");
        const header = document.createElement("tr");
        ["K", "Student", "Prize", "Overwrite"].forEach((text) => {
          const th = document.createElement("th");
          th.textContent = text;
          header.appendChild(th);
        });
        thead.appendChild(header);

        const tbody = document.createElement("tbody");
        const prizes = getPrizes();

        assignments.forEach((entry) => {
          const row = document.createElement("tr");

          const classCell = document.createElement("td");
          classCell.className = "tp-k-cell";
          const entryClass = cleanClassLabel(entry.class || entry.k || "", classOptions);
          const classSelectForRow = document.createElement("select");
          classSelectForRow.className = "tp-row-select";
          classSelectForRow.replaceChildren(...optionList(classOptions, entryClass));

          const classSaveButton = document.createElement("button");
          classSaveButton.className = "tp-small-button";
          classSaveButton.type = "button";
          classSaveButton.textContent = "Save K";
          classSaveButton.addEventListener("click", () => {
            model.send({
              type: "update_student_class",
              old_class_name: entryClass,
              new_class_name: classSelectForRow.value,
              student: entry.student
            });
          });
          classCell.append(classSelectForRow, classSaveButton);

          const studentCell = document.createElement("td");
          studentCell.textContent = entry.student || "";

          const prizeCell = document.createElement("td");
          prizeCell.textContent = entry.prize || "";

          const overwriteCell = document.createElement("td");
          const rowSelect = document.createElement("select");
          rowSelect.className = "tp-row-select";
          rowSelect.replaceChildren(...optionList(prizes, entry.prize));

          const saveButton = document.createElement("button");
          saveButton.className = "tp-small-button";
          saveButton.type = "button";
          saveButton.textContent = "Save";
          saveButton.addEventListener("click", () => {
            model.send({
              type: "overwrite_prize",
              class_name: entryClass,
              student: entry.student,
              prize: rowSelect.value
            });
          });

          overwriteCell.append(rowSelect, saveButton);
          row.append(classCell, studentCell, prizeCell, overwriteCell);
          tbody.appendChild(row);
        });

        table.append(thead, tbody);
        tableWrap.appendChild(table);
        updateButton();
      }

      function resetRotationForNewPrizes() {
        rotation = 0;
        spinner.style.transition = "none";
        spinner.style.transform = "rotate(0deg)";
        redraw();
        renderTable();
      }

      classSelect.addEventListener("change", () => {
        model.set("selected_class", cleanClassLabel(classSelect.value, classOptions));
        model.save_changes();
        updateButton();
      });

      nameInput.addEventListener("input", () => {
        const cleanName = cleanText(nameInput.value);
        model.set("current_student", cleanName);
        model.save_changes();
        updateButton();
        hint.textContent = cleanName ? "Click SPIN in the middle of the wheel." : "Enter a student name before spinning.";
      });

      button.addEventListener("click", () => {
        const prizes = getPrizes();
        const student = cleanText(nameInput.value);
        const className = cleanClassLabel(classSelect.value, classOptions);
        if (!student) {
          result.textContent = "Enter a student name first.";
          return;
        }
        if (locked || model.get("spinning") || prizes.length < 2) {
          return;
        }
        locked = true;
        result.textContent = `Spinning for ${className} - ${student}...`;
        updateButton();
        model.send({ type: "request_spin", class_name: className, student });
      });

      overwriteCurrentButton.addEventListener("click", () => {
        const student = cleanText(nameInput.value);
        const className = cleanClassLabel(classSelect.value, classOptions);
        const prize = manualSelect.value;
        if (!student || !prize) {
          status.textContent = "Choose a class, enter a student name, and choose a prize first.";
          return;
        }
        model.send({ type: "overwrite_prize", class_name: className, student, prize });
      });

      clearCurrentButton.addEventListener("click", () => {
        status.textContent = "Clearing the current on-screen list...";
        model.send({ type: "clear_current_assignments" });
      });

      function resetClearAllButton() {
        clearAllArmed = false;
        clearAllButton.textContent = "Clear Entire List";
      }

      clearAllButton.addEventListener("click", () => {
        if (!clearAllArmed) {
          clearAllArmed = true;
          clearAllButton.textContent = "Click Again to Clear File";
          status.textContent = "Click Clear Entire List one more time to erase the saved file.";
          if (clearAllTimer) window.clearTimeout(clearAllTimer);
          clearAllTimer = window.setTimeout(resetClearAllButton, 5000);
          return;
        }

        if (clearAllTimer) window.clearTimeout(clearAllTimer);
        resetClearAllButton();
        status.textContent = "Clearing the tracker and saved file...";
        model.send({ type: "clear_all_assignments" });
      });

      function handleCustomMessage(msg) {
        if (!msg || msg.type !== "spin_result") {
          return;
        }

        const count = msg.count;
        const index = msg.index;
        const segment = 360 / count;
        const desiredMod = normalizeDegrees(360 - ((index + 0.5) * segment));
        const currentMod = normalizeDegrees(rotation);
        const delta = normalizeDegrees(desiredMod - currentMod) + (360 * 6);
        rotation += delta;

        spinner.style.transition = "transform 5.2s cubic-bezier(0.12, 0.83, 0.18, 1)";
        spinner.style.transform = `rotate(${rotation}deg)`;

        let finished = false;
        const finish = () => {
          if (finished) return;
          finished = true;
          if (fallbackTimer) window.clearTimeout(fallbackTimer);
          result.textContent = `${msg.class_name}: ${msg.student} won ${msg.prize}`;
          locked = false;
          updateButton();
          model.send({ type: "spin_complete", spin_id: msg.spin_id });
        };

        spinner.addEventListener("transitionend", finish, { once: true });
        fallbackTimer = window.setTimeout(finish, 6200);
      }

      model.on("msg:custom", handleCustomMessage);
      model.on("change:assignments", renderTable);
      model.on("change:prizes", resetRotationForNewPrizes);
      model.on("change:title", () => {
        title.textContent = model.get("title") || "PineBrook IGNITE Prize Wheel";
      });
      model.on("change:spinning", updateButton);
      model.on("change:status_message", () => {
        status.textContent = model.get("status_message") || "Ready.";
      });
      model.on("change:storage_path", () => {
        saveInfo.textContent = `Saved locally to: ${model.get("storage_path") || "student_prize_tracker.json"}`;
      });
      model.on("change:current_student", () => {
        const next = model.get("current_student") || "";
        if (nameInput.value !== next) {
          nameInput.value = next;
        }
        updateButton();
      });
      model.on("change:selected_class", () => {
        const nextClass = cleanClassLabel(model.get("selected_class"), classOptions);
        if (classSelect.value !== nextClass) {
          classSelect.value = nextClass;
        }
        updateButton();
      });

      const resizeObserver = new ResizeObserver(redraw);
      resizeObserver.observe(stage);
      redraw();
      renderTable();

      return () => {
        resizeObserver.disconnect();
        if (clearAllTimer) window.clearTimeout(clearAllTimer);
        model.off("msg:custom", handleCustomMessage);
      };
    }

    export default { render };
    """

    prizes = traitlets.List(trait=traitlets.Unicode(), default_value=[]).tag(sync=True)
    assignments = traitlets.List(trait=traitlets.Dict(), default_value=[]).tag(sync=True)
    storage_path = traitlets.Unicode("student_prize_tracker.json").tag(sync=True)
    class_options = traitlets.List(trait=traitlets.Unicode(), default_value=["K1", "K2", "K3", "K4", "K5"]).tag(sync=True)
    selected_class = traitlets.Unicode("K1").tag(sync=True)
    current_student = traitlets.Unicode("").tag(sync=True)
    selected = traitlets.Unicode("").tag(sync=True)
    selected_index = traitlets.Int(-1).tag(sync=True)
    last_student = traitlets.Unicode("").tag(sync=True)
    last_class = traitlets.Unicode("K1").tag(sync=True)
    spinning = traitlets.Bool(False).tag(sync=True)
    spin_id = traitlets.Int(0).tag(sync=True)
    title = traitlets.Unicode("PineBrook IGNITE Prize Wheel").tag(sync=True)
    seed = traitlets.Any(default_value=None).tag(sync=True)
    status_message = traitlets.Unicode("Ready.").tag(sync=True)

    def __init__(
        self,
        prizes: Iterable[str],
        seed=None,
        title="PineBrook IGNITE Prize Wheel",
        storage_path="student_prize_tracker.json",
        class_options: Iterable[str] = ("K1", "K2", "K3", "K4", "K5"),
        selected_class="K1",
        **kwargs,
    ):
        cleaned_prizes = [self._clean_text(item) for item in prizes if self._clean_text(item)]
        if len(cleaned_prizes) < 2:
            raise ValueError("Please provide at least two non-empty prize names.")

        cleaned_classes = [self._clean_class_label(item) for item in class_options if self._clean_class_label(item)]
        cleaned_classes = list(dict.fromkeys(cleaned_classes))
        if not cleaned_classes:
            cleaned_classes = ["K1", "K2", "K3", "K4", "K5"]

        selected_class = self._clean_class_label(selected_class)
        if selected_class not in cleaned_classes:
            selected_class = cleaned_classes[0]

        self._rng = random.Random(seed) if seed is not None else None
        self._pending = None

        storage_path = str(Path(storage_path).expanduser())
        loaded_assignments = kwargs.pop("assignments", None)
        if loaded_assignments is None:
            loaded_assignments = self._load_assignments_from_disk(storage_path, cleaned_prizes, cleaned_classes, selected_class)
        loaded_assignments = self._sanitize_assignments(loaded_assignments, cleaned_prizes, cleaned_classes, selected_class)

        super().__init__(
            prizes=cleaned_prizes,
            seed=seed,
            title=title,
            storage_path=storage_path,
            class_options=cleaned_classes,
            selected_class=selected_class,
            assignments=loaded_assignments,
            **kwargs,
        )
        self.on_msg(self._handle_frontend_message)

        if loaded_assignments:
            self.status_message = f"Loaded {len(loaded_assignments)} saved student(s) from {self.storage_path}."
        else:
            self.status_message = f"Ready. Saving to {self.storage_path}."

    @staticmethod
    def _clean_text(value) -> str:
        return " ".join(str(value).strip().split())

    @classmethod
    def _clean_class_label(cls, value) -> str:
        raw = cls._clean_text(value).upper().replace(" ", "")
        if not raw:
            return ""
        if raw.isdigit():
            return f"K{raw}"
        if raw.startswith("K") and raw[1:].isdigit():
            return f"K{raw[1:]}"
        return raw

    @classmethod
    def _class_number(cls, value) -> str:
        label = cls._clean_class_label(value)
        return label[1:] if label.startswith("K") else label

    @classmethod
    def _sanitize_assignments(
        cls,
        entries,
        prizes: Iterable[str],
        class_options: Iterable[str],
        default_class="K1",
    ):
        """Clean loaded/supplied tracker rows and keep one row per class/student."""
        prize_list = [cls._clean_text(prize) for prize in prizes if cls._clean_text(prize)]
        class_list = [cls._clean_class_label(item) for item in class_options if cls._clean_class_label(item)]
        default_class = cls._clean_class_label(default_class) or (class_list[0] if class_list else "K1")

        by_class_student = {}
        order = []

        for raw in entries or []:
            if not isinstance(raw, dict):
                continue

            class_name = cls._clean_class_label(raw.get("class", raw.get("class_name", raw.get("k", default_class))))
            if class_name not in class_list:
                class_name = default_class

            student = cls._clean_text(raw.get("student", ""))
            prize = cls._clean_text(raw.get("prize", ""))
            if not student or not prize:
                continue

            key = (class_name, student.casefold())
            if key not in by_class_student:
                order.append(key)

            if prize in prize_list:
                prize_index = prize_list.index(prize)
            else:
                try:
                    prize_index = int(raw.get("prize_index", -1))
                except (TypeError, ValueError):
                    prize_index = -1

            updated_at = cls._clean_text(raw.get("updated_at", "")) or datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            by_class_student[key] = {
                "class": class_name,
                "k": cls._class_number(class_name),
                "student": student,
                "prize": prize,
                "prize_index": prize_index,
                "updated_at": updated_at,
            }

        return [by_class_student[key] for key in order]

    @classmethod
    def _load_assignments_from_disk(
        cls,
        storage_path: str,
        prizes: Iterable[str],
        class_options: Iterable[str],
        default_class="K1",
    ):
        """Read saved tracker data from JSON if the file exists."""
        path = Path(storage_path).expanduser()
        if not path.exists():
            return []

        try:
            data = json.loads(path.read_text(encoding="utf-8"))
        except Exception as exc:
            print(f"Could not read saved tracker file {path}: {exc}")
            return []

        if isinstance(data, dict):
            raw_assignments = data.get("assignments", [])
        elif isinstance(data, list):
            raw_assignments = data
        else:
            raw_assignments = []

        return cls._sanitize_assignments(raw_assignments, prizes, class_options, default_class)

    def _save_assignments_to_disk(self):
        """Save the current tracker data to JSON."""
        path = Path(self.storage_path).expanduser()
        try:
            path.parent.mkdir(parents=True, exist_ok=True)
            cleaned_assignments = self._sanitize_assignments(
                self.assignments,
                self.prizes,
                self.class_options,
                self.selected_class,
            )
            payload = {
                "version": 2,
                "updated_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                "class_options": list(self.class_options),
                "assignments": cleaned_assignments,
            }
            path.write_text(json.dumps(payload, indent=2), encoding="utf-8")
        except Exception as exc:
            self.status_message = f"Could not save tracker to {path}: {exc}"

    def _pick_index(self) -> int:
        if self._rng is not None:
            return self._rng.randrange(len(self.prizes))
        return secrets.randbelow(len(self.prizes))

    def _record_assignment(self, student: str, prize: str, prize_index: int | None = None, class_name: str | None = None):
        student = self._clean_text(student)
        prize = self._clean_text(prize)
        class_name = self._clean_class_label(class_name or self.selected_class)
        if class_name not in self.class_options:
            class_name = self.class_options[0]

        if not student:
            self.status_message = "Please enter a student name first."
            return
        if prize not in self.prizes:
            raise ValueError(f"{prize!r} is not in this wheel's prize list.")

        prize_index = self.prizes.index(prize) if prize_index is None else prize_index
        now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        next_assignments = [dict(item) for item in self.assignments]
        match_index = None

        for index, entry in enumerate(next_assignments):
            entry_class = self._clean_class_label(entry.get("class", entry.get("k", self.selected_class)))
            entry_student = self._clean_text(entry.get("student", ""))
            if entry_class == class_name and entry_student.casefold() == student.casefold():
                match_index = index
                break

        new_entry = {
            "class": class_name,
            "k": self._class_number(class_name),
            "student": student,
            "prize": prize,
            "prize_index": prize_index,
            "updated_at": now,
        }

        if match_index is None:
            next_assignments.append(new_entry)
            self.status_message = f"Added {class_name} - {student}: {prize}"
        else:
            previous_prize = self._clean_text(next_assignments[match_index].get("prize", "")) or "None"
            next_assignments[match_index] = new_entry
            self.status_message = (
                f"Updated {class_name} - {student}:\n"
                f"Prize Before: {previous_prize}\n"
                f"Prize Now: {prize}\n"
                "You can select which prize to proceed with in the tracker."
            )

        self.assignments = self._sanitize_assignments(next_assignments, self.prizes, self.class_options, class_name)
        self._save_assignments_to_disk()
        self.selected_class = class_name
        self.current_student = student
        self.last_class = class_name
        self.last_student = student
        self.selected = prize
        self.selected_index = prize_index

    def _update_assignment_class(self, student: str, old_class_name: str, new_class_name: str):
        """Move one tracked student record from one K/class to another and save it."""
        student = self._clean_text(student)
        old_class_name = self._clean_class_label(old_class_name or self.selected_class)
        new_class_name = self._clean_class_label(new_class_name or old_class_name)

        if not student:
            self.status_message = "Choose a student record before updating K."
            return
        if old_class_name not in self.class_options:
            self.status_message = "Choose a valid original K before updating."
            return
        if new_class_name not in self.class_options:
            self.status_message = "Choose a valid new K before updating."
            return
        if old_class_name == new_class_name:
            self.status_message = f"{student} is already in {new_class_name}. No K change needed."
            return

        source_entry = None
        remaining = []
        for entry in self.assignments:
            entry_class = self._clean_class_label(entry.get("class", entry.get("k", self.selected_class)))
            entry_student = self._clean_text(entry.get("student", ""))
            if source_entry is None and entry_class == old_class_name and entry_student.casefold() == student.casefold():
                source_entry = dict(entry)
            else:
                remaining.append(dict(entry))

        if source_entry is None:
            self.status_message = f"No matching record found for {old_class_name} - {student}."
            return

        moved_student = self._clean_text(source_entry.get("student", student)) or student
        source_entry["class"] = new_class_name
        source_entry["k"] = self._class_number(new_class_name)
        source_entry["student"] = moved_student
        source_entry["updated_at"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

        next_assignments = []
        inserted = False
        replaced_existing = False
        for entry in remaining:
            entry_class = self._clean_class_label(entry.get("class", entry.get("k", self.selected_class)))
            entry_student = self._clean_text(entry.get("student", ""))
            if entry_class == new_class_name and entry_student.casefold() == moved_student.casefold():
                if not inserted:
                    next_assignments.append(source_entry)
                    inserted = True
                replaced_existing = True
            else:
                next_assignments.append(entry)

        if not inserted:
            next_assignments.append(source_entry)

        self.assignments = self._sanitize_assignments(next_assignments, self.prizes, self.class_options, new_class_name)
        self._save_assignments_to_disk()
        self.selected_class = new_class_name
        self.current_student = moved_student
        self.last_class = new_class_name
        self.last_student = moved_student
        note = " Existing record in the new K was replaced." if replaced_existing else ""
        self.status_message = f"Updated {moved_student}'s K: {old_class_name} -> {new_class_name}.{note}"

    def _handle_frontend_message(self, widget, content, buffers):
        msg_type = content.get("type")

        if msg_type == "request_spin":
            if self.spinning:
                return

            student = self._clean_text(content.get("student") or self.current_student)
            class_name = self._clean_class_label(content.get("class_name") or self.selected_class)
            if class_name not in self.class_options:
                class_name = self.class_options[0]

            if not student:
                self.status_message = "Please enter a student name first."
                return

            self.selected_class = class_name
            self.current_student = student
            self.spinning = True
            self.spin_id += 1
            index = self._pick_index()
            prize = self.prizes[index]
            self._pending = {
                "spin_id": self.spin_id,
                "class_name": class_name,
                "student": student,
                "index": index,
                "prize": prize,
            }
            self.status_message = f"Spinning for {class_name} - {student}..."
            self.send({
                "type": "spin_result",
                "spin_id": self.spin_id,
                "class_name": class_name,
                "student": student,
                "index": index,
                "prize": prize,
                "count": len(self.prizes),
            })

        elif msg_type == "spin_complete":
            if not self._pending:
                return
            if content.get("spin_id") != self._pending["spin_id"]:
                return

            pending = self._pending
            self._record_assignment(
                pending["student"],
                pending["prize"],
                pending["index"],
                pending["class_name"],
            )
            self.spinning = False
            self._pending = None

        elif msg_type == "overwrite_prize":
            student = self._clean_text(content.get("student") or self.current_student)
            class_name = self._clean_class_label(content.get("class_name") or self.selected_class)
            if class_name not in self.class_options:
                class_name = self.class_options[0]
            prize = self._clean_text(content.get("prize"))
            if not student:
                self.status_message = "Enter a student name before overwriting."
                return
            if prize not in self.prizes:
                self.status_message = "Choose a valid prize before overwriting."
                return
            self._record_assignment(student, prize, class_name=class_name)

        elif msg_type == "update_student_class":
            self._update_assignment_class(
                content.get("student"),
                content.get("old_class_name"),
                content.get("new_class_name"),
            )

        elif msg_type == "clear_current_assignments":
            self.clear_current_list()

        elif msg_type == "clear_all_assignments":
            self.clear_entire_list()

    def clear_current_list(self):
        """Clear only the current in-memory/on-screen tracker.

        This does NOT change the saved JSON file. Rerunning the notebook or calling
        reload_from_disk() will bring the saved list back.
        """
        self.assignments = []
        self.selected = ""
        self.selected_index = -1
        self.last_student = ""
        self.last_class = self.selected_class
        self.spinning = False
        self._pending = None
        self.status_message = "Current on-screen list cleared. Saved local file was not changed."

    def clear_entire_list(self):
        """Clear the current tracker and overwrite the saved JSON file with 0 student records."""
        path = Path(self.storage_path).expanduser()
        empty_payload = {
            "version": 2,
            "updated_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "class_options": list(self.class_options),
            "assignments": [],
        }
        try:
            path.parent.mkdir(parents=True, exist_ok=True)
            path.write_text(json.dumps(empty_payload, indent=2), encoding="utf-8")
        except Exception as exc:
            self.status_message = f"Could not clear saved tracker file {path}: {exc}"
            return

        self.assignments = []
        self.selected = ""
        self.selected_index = -1
        self.last_student = ""
        self.last_class = self.selected_class
        self.spinning = False
        self._pending = None
        self.status_message = f"Entire student prize tracker cleared. Saved file now has 0 student records: {path}."

    # Backwards-friendly alias from the earlier notebook version.
    def clear_students(self):
        """Clear the current tracker and saved JSON file."""
        self.clear_entire_list()

    def set_student_prize(self, student: str, prize: str, class_name: str = "K1"):
        """Set or overwrite one student's prize from Python."""
        self._record_assignment(student, prize, class_name=class_name)

    def save_now(self):
        """Manually save the current tracker to the local JSON file."""
        self._save_assignments_to_disk()
        path = Path(self.storage_path).expanduser()
        self.status_message = f"Saved tracker to {path}."
        return str(path.resolve())

    def reload_from_disk(self):
        """Reload the tracker from the local JSON file."""
        self.assignments = self._load_assignments_from_disk(
            self.storage_path,
            self.prizes,
            self.class_options,
            self.selected_class,
        )
        self.status_message = f"Reloaded {len(self.assignments)} student(s) from {self.storage_path}."
        return self.tracking_table()

    def reset(self):
        """Unlock the wheel and clear the last selection, but keep the tracker."""
        self.selected = ""
        self.selected_index = -1
        self.spinning = False
        self._pending = None
        self.status_message = "Ready."

    def tracking_table(self):
        """Return the current tracker as a pandas DataFrame."""
        import pandas as pd
        df = pd.DataFrame(self.assignments, columns=["class", "k", "student", "prize", "prize_index", "updated_at"])
        if not df.empty:
            df = df.rename(columns={"class": "class_name", "k": "K"})
        return df


def show_student_prize_wheel(
    prizes,
    seed=None,
    title="PineBrook IGNITE Prize Wheel",
    storage_path="student_prize_tracker.json",
    class_options=("K1", "K2", "K3", "K4", "K5"),
):
    """Create and display the student-tracking prize wheel."""
    wheel = StudentPrizeWheel(
        prizes=prizes,
        seed=seed,
        title=title,
        storage_path=storage_path,
        class_options=class_options,
    )
    display(wheel)
    return wheel

In [ ]:
# Change this list to whatever prizes you want.
# The wheel uses whatever list YOU pass in here.
my_prizes = [
    "UNO Cards",
    "Sticker",
    # "Headphones",
    "Big Cup",
    "Small Cup",
    "Soft Hand Ball",
    "Sticky Hand Ball",
    "Drawing Pad",
    "Shoot the Moon Game",
]

# This JSON file is created locally next to your notebook/kernel working folder.
# When you run this notebook again, the wheel reloads the saved students and prizes from this file.
tracker_file = "student_prize_tracker.json"

wheel = show_student_prize_wheel(
    my_prizes,
    title="PineBrook IGNITE Prize Wheel",
    storage_path=tracker_file,
    class_options=("K1", "K2", "K3", "K4", "K5"),
    seed=None,  # use a number like 42 if you want repeatable results
)


In [ ]:
# Optional Python controls you can run anytime.

# See the current tracker as a table:
wheel.tracking_table()

# Clear only the current on-screen/in-memory list.
# This does not change the saved JSON file.
# wheel.clear_current_list()

# Clear the entire tracker, including the saved JSON file.
# wheel.clear_entire_list()

# Reload the saved local file:
# wheel.reload_from_disk()

# Save manually if needed. The wheel also saves automatically after every spin or overwrite.
# wheel.save_now()

# Manually set or overwrite one student's prize from Python.
# wheel.set_student_prize("Student Name", "Sticker", class_name="K1")
